# Rua Avisada — Painel do Data Warehouse

**Tema:** a última milha do alerta de chuva em São Paulo — onde o terreno favorece alagamento **e** a drenagem está atrasada.
**Período:** temporada de chuvas out/2025–mar/2026 · **Território:** 96 distritos.

**Perguntas analíticas**
1. Quais distritos são propensos a alagar **e** estão com a drenagem atrasada? (página 2)
2. O atendimento chega antes da chuva? (página 1 — chuva × backlog)
3. Onde se alaga e não se reclama? (página 3)

**Como rodar:** abra este notebook na pasta `dashboard/` do projeto e use *Run All*.
Requisitos: `pip install pandas plotly nbformat jupyter`.
No fim, o painel completo é salvo em `dashboard/painel_rua_avisada.html` (abre no navegador, interativo).

In [ ]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Funciona rodando de dentro de dashboard/ ou da pasta do projeto
BASE = Path("..") if Path.cwd().name == "dashboard" else Path(".")
MODELO = BASE / "data" / "modelo"
STAGING = BASE / "data" / "staging"
SAIDA_HTML = BASE / "dashboard" / "painel_rua_avisada.html"

# Mesma escolha do script 07 (BASE_SUJEIRA): "taxa_pendencia" ou "divida_10mil_hab"
BASE_SUJEIRA = "taxa_pendencia"
COLUNA_SUJEIRA = {"taxa_pendencia": "taxa_pendencia_pct", "divida_10mil_hab": "divida_drenagem_10mil_hab"}[BASE_SUJEIRA]
NOME_SUJEIRA = {"taxa_pendencia": "Pendência na véspera da chuva (%)",
                "divida_10mil_hab": "Dívida (pend./10 mil hab.)"}[BASE_SUJEIRA]

CORES_QUADRANTE = {"zona vermelha": "#C0392B", "zeladoria": "#E67E22",
                   "estrutural": "#2E86C1", "fora do modelo": "#AAB7B8"}
ORDEM_QUADRANTE = list(CORES_QUADRANTE)
FIGURAS = []   # cada visual entra aqui para o HTML final


def mostrar(fig, titulo_html):
    """Mostra a figura no notebook e guarda para o HTML do painel."""
    fig.update_layout(template="plotly_white", margin=dict(l=20, r=20, t=60, b=20))
    FIGURAS.append((titulo_html, fig))
    fig.show()


def br(n, casas=0):
    """12345.6 -> '12.345,6'"""
    return f"{n:,.{casas}f}".replace(",", "X").replace(".", ",").replace("X", ".")

## 1. O modelo Snowflake (CSV)

Cada tabela do DW é um CSV em `data/modelo/`. Abaixo, o tamanho de cada uma e os **joins pelas chaves**,
exatamente como no diagrama: as fatos ligam no distrito, que liga na subprefeitura, que liga na zona.

In [ ]:
tabelas = {nome: pd.read_csv(MODELO / f"{nome}.csv") for nome in [
    "fato_distrito_dia", "fato_solicitacao", "agg_indicadores_distrito",
    "dim_distrito", "dim_subprefeitura", "dim_zona", "dim_estacao_inmet",
    "dim_data", "dim_servico", "dim_assunto", "dim_tema", "dim_status"]}
pd.DataFrame({"tabela": list(tabelas), "linhas": [len(t) for t in tabelas.values()],
              "colunas": [t.shape[1] for t in tabelas.values()]})

In [ ]:
t = tabelas
# Território (Snowflake): distrito -> subprefeitura -> zona
territorio = (t["dim_distrito"]
              .merge(t["dim_subprefeitura"], on="sk_subprefeitura", how="left")
              .merge(t["dim_zona"], on="sk_zona", how="left"))

# Indicadores por distrito + território
distritos = territorio.merge(t["agg_indicadores_distrito"], on="sk_distrito", how="left",
                             suffixes=("", "_agg"))

# Fato distrito x dia + território + calendário
dia = (t["fato_distrito_dia"]
       .merge(territorio[["sk_distrito", "nm_distrito", "nm_subprefeitura", "nm_zona"]], on="sk_distrito")
       .merge(t["dim_data"][["sk_data", "data", "nome_mes", "temporada_chuva"]], on="sk_data"))
dia["data"] = pd.to_datetime(dia["data"])

# Fato solicitação + hierarquia de serviço (serviço -> assunto -> tema) + status
solicitacoes = (t["fato_solicitacao"]
                .merge(t["dim_servico"], on="sk_servico")
                .merge(t["dim_assunto"], on="sk_assunto")
                .merge(t["dim_tema"], on="sk_tema")
                .merge(t["dim_status"], on="sk_status"))

kpis = pd.read_csv(STAGING / "kpis_cidade.csv").set_index("kpi")["valor"]
geojson = json.loads((STAGING / "distritos_wgs84.geojson").read_text(encoding="utf-8"))
print(f"{len(distritos)} distritos · {len(dia):,} linhas distrito×dia · {len(solicitacoes):,} chamados".replace(",", "."))

---
# Página 1 — Visão geral

### KPIs da temporada
- **Chamados de drenagem:** pedidos de manutenção (bueiro, boca de lobo, galeria, microdrenagem, córrego) não cancelados.
- **Tempo mediano:** metade dos chamados finalizados foi resolvida em até esse número de dias.
- **Pico do backlog:** maior número de chamados pendentes na cidade em um mesmo dia.
- **Pendência na véspera da chuva:** dos chamados abertos até a véspera de cada dia de chuva forte (≥ 20 mm), quantos % ainda estavam pendentes (média dos distritos). Mede o atraso do serviço.
- **Dívida de drenagem:** chamados pendentes na véspera de dias de chuva forte, por 10 mil habitantes.

In [ ]:
def k(nome):
    return float(kpis[nome])

pico = str(int(k("data_backlog_pico")))
cartoes = [
    ("Chamados<br>de drenagem", k("chamados_drenagem_temporada")),
    ("Finalizados<br>(%)", k("pct_finalizados")),
    ("Tempo mediano<br>(dias)", k("tempo_mediano_dias")),
    (f"Pico do backlog<br>({pico[6:]}/{pico[4:6]})", k("backlog_pico")),
    ("Pendência na véspera<br>da chuva (%)", k("taxa_pendencia_media_pct")),
    ("Dívida<br>(pend./10 mil hab.)", k("divida_drenagem_cidade_10mil_hab")),
]
fig = go.Figure()
for i, (titulo, valor) in enumerate(cartoes):
    fig.add_trace(go.Indicator(mode="number", value=valor,
                               title={"text": titulo, "font": {"size": 14}},
                               number={"valueformat": ",.1f" if valor % 1 else ",.0f", "font": {"size": 38}},
                               domain={"row": 0, "column": i}))
fig.update_layout(grid={"rows": 1, "columns": len(cartoes)}, height=230, separators=",.",
                  title="KPIs da temporada de chuvas (out/2025–mar/2026)")
mostrar(fig, "KPIs")

### Mapa da drenagem atrasada
Cor = indicador de "sujeira" usado nos quadrantes (por padrão, % dos chamados ainda pendentes na véspera de chuva forte).
Passe o mouse para ver subprefeitura, zona, quadrante, dívida por 10 mil hab. e tempo de atendimento.

In [ ]:
fig = px.choropleth(
    distritos, geojson=geojson, locations="sk_distrito", featureidkey="properties.cd_distrito",
    color=COLUNA_SUJEIRA, color_continuous_scale="OrRd",
    hover_name="nm_distrito",
    hover_data={"sk_distrito": False, "nm_subprefeitura": True, "nm_zona": True, "quadrante": True,
                "tempo_mediano_dias": True, "taxa_pendencia_pct": ":.1f", "divida_drenagem_10mil_hab": ":.2f"},
    labels={"divida_drenagem_10mil_hab": "Dívida (pend./10 mil hab.)", "taxa_pendencia_pct": "Pendência (%)",
            "nm_subprefeitura": "Subprefeitura", "nm_zona": "Zona", "quadrante": "Quadrante",
            "tempo_mediano_dias": "Tempo mediano (dias)", COLUNA_SUJEIRA: NOME_SUJEIRA},
)
fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(width=850, height=650, title=f"{NOME_SUJEIRA} por distrito",
                  coloraxis_colorbar=dict(title="", x=0.92))
mostrar(fig, "Mapa da drenagem atrasada")

### O atendimento chega antes da chuva? — chuva × backlog
Barras = chuva diária (média das estações dos distritos selecionados). Linha = chamados de drenagem pendentes no fim do dia.
Use o **menu** para filtrar por zona. Se a linha sobe logo depois das barras altas e demora a descer,
o atendimento está correndo atrás da chuva.

In [ ]:
def serie(df):
    return df.groupby("data").agg(chuva_mm=("chuva_mm", "mean"),
                                  backlog=("backlog_drenagem", "sum")).reset_index()

opcoes = {"Cidade": dia}
opcoes.update({z: dia[dia["nm_zona"] == z] for z in sorted(dia["nm_zona"].dropna().unique())})

fig = go.Figure()
for i, (nome, df) in enumerate(opcoes.items()):
    s = serie(df)
    visivel = i == 0
    fig.add_trace(go.Bar(x=s["data"], y=s["chuva_mm"], name="Chuva (mm)", marker_color="#5DADE2",
                         yaxis="y2", opacity=0.6, visible=visivel))
    fig.add_trace(go.Scatter(x=s["data"], y=s["backlog"], name="Backlog (pendentes)",
                             line=dict(color="#C0392B", width=2.5), visible=visivel))
botoes = []
for i, nome in enumerate(opcoes):
    vis = [False] * (2 * len(opcoes))
    vis[2 * i] = vis[2 * i + 1] = True
    botoes.append(dict(label=nome, method="update", args=[{"visible": vis}, {"title": f"Chuva × backlog — {nome}"}]))
fig.update_layout(
    title=dict(text="Chuva × backlog — Cidade", x=0), height=500, hovermode="x unified",
    yaxis=dict(title="Chamados pendentes"),
    yaxis2=dict(title="Chuva (mm/dia)", overlaying="y", side="right", showgrid=False),
    updatemenus=[dict(buttons=botoes, direction="down", x=1, y=1.18, xanchor="right", yanchor="top")],
    annotations=[dict(text="Zona:", x=0.86, y=1.16, xref="paper", yref="paper", showarrow=False, xanchor="right")],
    legend=dict(orientation="h", y=-0.15),
)
mostrar(fig, "Chuva × backlog")

---
# Página 2 — Prevenção

### Matriz de quadrantes
- **Eixo X — propensão do terreno (0–100):** posição do distrito em área plana, declividade, altitude e proximidade de córrego.
- **Eixo Y — sujeira (0–100):** posição do distrito no indicador de drenagem atrasada (por padrão, a % de chamados pendentes na véspera de chuva forte — mede o atraso, não o volume de pedidos).
- Corte em 50 nos dois eixos. **Zona vermelha** = propenso + sujo: é onde a limpeza preventiva rende mais.
  **Zeladoria** = terreno não favorece, mas a drenagem está atrasada. **Estrutural** = terreno favorece, drenagem em dia.
- Tamanho da bolha = população.

In [ ]:
fig = px.scatter(
    distritos, x="indice_propensao", y="score_sujeira", color="quadrante", size="populacao",
    color_discrete_map=CORES_QUADRANTE, category_orders={"quadrante": ORDEM_QUADRANTE},
    hover_name="nm_distrito", size_max=40,
    hover_data={"populacao": ":,", "taxa_pendencia_pct": ":.1f", "divida_drenagem_10mil_hab": ":.2f", "nm_zona": True},
    labels={"indice_propensao": "Propensão do terreno (0–100)", "score_sujeira": "Sujeira — drenagem atrasada (0–100)",
            "taxa_pendencia_pct": "Pendência (%)",
            "quadrante": "Quadrante", "populacao": "População", "nm_zona": "Zona",
            "divida_drenagem_10mil_hab": "Dívida (pend./10 mil hab.)"},
)
fig.add_vline(x=50, line_dash="dash", line_color="gray")
fig.add_hline(y=50, line_dash="dash", line_color="gray")
for x, y, texto in [(75, 102, "ZONA VERMELHA"), (25, 102, "ZELADORIA"), (75, -4, "ESTRUTURAL"), (25, -4, "FORA DO MODELO")]:
    fig.add_annotation(x=x, y=y, text=texto, showarrow=False, font=dict(size=12, color="gray"))
fig.update_layout(height=620, title="Matriz de quadrantes: propensão do terreno × drenagem atrasada",
                  xaxis=dict(range=[-2, 102]), yaxis=dict(range=[-8, 106]))
mostrar(fig, "Matriz de quadrantes")

In [ ]:
fig = px.choropleth(
    distritos, geojson=geojson, locations="sk_distrito", featureidkey="properties.cd_distrito",
    color="quadrante", color_discrete_map=CORES_QUADRANTE, category_orders={"quadrante": ORDEM_QUADRANTE},
    hover_name="nm_distrito", hover_data={"sk_distrito": False, "prioridade": True, "nm_subprefeitura": True},
    labels={"quadrante": "Quadrante", "prioridade": "Prioridade", "nm_subprefeitura": "Subprefeitura"},
)
fig.update_geos(fitbounds="locations", visible=False)
fig.update_layout(width=850, height=650, title="Quadrantes no mapa",
                  legend=dict(x=0.02, y=0.98, bgcolor="rgba(255,255,255,0.8)", title="Quadrante"))
mostrar(fig, "Mapa dos quadrantes")

### Lista da zona vermelha (prioridade de limpeza preventiva)
Prioridade = média da propensão e da sujeira. É a lista que a prefeitura receberia antes de um alerta de chuva.

In [ ]:
verm = (distritos[distritos["quadrante"] == "zona vermelha"]
        .sort_values("prioridade", ascending=False).head(15).reset_index(drop=True))
colunas = {"nm_distrito": "Distrito", "nm_subprefeitura": "Subprefeitura", "indice_propensao": "Propensão",
           "score_sujeira": "Sujeira", "taxa_pendencia_pct": "Pendência (%)",
           "divida_drenagem_10mil_hab": "Dívida (pend./10 mil hab.)",
           "tempo_mediano_dias": "Tempo mediano (dias)", "pct_idosos": "% idosos"}
tabela = verm[list(colunas)].rename(columns=colunas)
tabela.insert(0, "#", range(1, len(tabela) + 1))
fig = go.Figure(go.Table(
    header=dict(values=list(tabela.columns), fill_color="#C0392B", font=dict(color="white"), align="left"),
    cells=dict(values=[tabela[c] for c in tabela.columns], align="left", fill_color="#FDEDEC"),
    columnwidth=[0.4, 1.6, 2.2, 0.9, 0.9, 1, 1.2, 1.1, 0.9],
))
fig.update_layout(height=520, title=f"Zona vermelha — {int(k('distritos_zona_vermelha'))} distritos, "
                                    f"{br(k('populacao_zona_vermelha'))} moradores ({br(k('pop_60_mais_zona_vermelha'))} com 60+)")
mostrar(fig, "Lista da zona vermelha")

---
# Página 3 — Subnotificação (jan–mar/2026)
Índice de subnotificação = ocorrências de alagamento/inundação registradas pela Defesa Civil ÷ (chamados de drenagem no SP156 + 1).
Alto = **alaga e não se reclama**: lugares onde o aviso por rua (WhatsApp, guardião, régua no poste) faz mais diferença.
*Limitação:* só temos ocorrências da Defesa Civil a partir de jan/2026.

In [ ]:
sub = (distritos[distritos["ocorrencias_dc_jan_mar"] > 0]
       .sort_values("indice_subnotificacao", ascending=True).tail(15))
fig = px.bar(sub, x="indice_subnotificacao", y="nm_distrito", orientation="h", color="nm_zona",
             hover_data={"ocorrencias_dc_jan_mar": True, "chamados_drenagem_jan_mar": True},
             labels={"indice_subnotificacao": "Ocorrências DC por chamado", "nm_distrito": "",
                     "nm_zona": "Zona", "ocorrencias_dc_jan_mar": "Ocorrências DC",
                     "chamados_drenagem_jan_mar": "Chamados SP156"})
fig.update_layout(height=550, title="Onde se alaga e não se reclama (top 15)",
                  yaxis=dict(categoryorder="total ascending"),
                  legend=dict(orientation="h", y=-0.12, title="Zona"))
mostrar(fig, "Subnotificação")

---
## Salvar o painel em HTML
Gera `dashboard/painel_rua_avisada.html` com todos os visuais interativos (para o .zip e para os prints).

In [ ]:
partes = ["<html><head><meta charset='utf-8'><title>Rua Avisada — Painel</title></head>",
          "<body style='font-family:Arial;max-width:1200px;margin:auto'>",
          "<h1>Rua Avisada — Painel do DW</h1>",
          "<p>Temporada de chuvas out/2025–mar/2026 · 96 distritos de São Paulo · "
          "Fontes: SP156, GeoSampa, Defesa Civil (COMDEC), INMET, TOPODATA/INPE, Censo 2022 (IBGE/TabNet SMS).</p>"]
for i, (titulo, fig) in enumerate(FIGURAS):
    partes.append(f"<h2>{titulo}</h2>")
    partes.append(fig.to_html(full_html=False, include_plotlyjs=(i == 0)))  # plotly embutido 1 vez (abre offline)
partes.append("</body></html>")
SAIDA_HTML.parent.mkdir(parents=True, exist_ok=True)
SAIDA_HTML.write_text("\n".join(partes), encoding="utf-8")
print(f"Painel salvo em {SAIDA_HTML.resolve()}")